# Leitura da base SICOR Cerrado — 2021 a 2025

Este notebook monta o Google Drive, lê o arquivo `sicor_cerrado_2021_2025.csv` e apresenta uma conferência inicial da base. A execução não altera o arquivo original.

## 1. Montar o Google Drive e importar as bibliotecas

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

Mounted at /content/drive


## 2. Definir e conferir o caminho do arquivo

No Google Colab, o caminho do Windows `G:\Meu Drive\...` corresponde a `/content/drive/MyDrive/...`.

In [2]:
ARQUIVO_CSV = Path(
    '/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/segundo/'
    'sicor_cerrado_2021_2025/sicor_cerrado_2021_2025.csv'
)

if not ARQUIVO_CSV.exists():
    raise FileNotFoundError(
        'Arquivo não encontrado. Confira se o Google Drive foi montado e se o caminho está correto:\n'
        f'{ARQUIVO_CSV}'
    )

tamanho_gb = ARQUIVO_CSV.stat().st_size / (1024 ** 3)
print('Arquivo encontrado:')
print(ARQUIVO_CSV)
print(f'Tamanho: {tamanho_gb:.2f} GB')

Arquivo encontrado:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/segundo/sicor_cerrado_2021_2025/sicor_cerrado_2021_2025.csv
Tamanho: 0.31 GB


## 3. Ler o CSV completo

A leitura usa o padrão empregado na consolidação da Base02: separador `;` e codificação `utf-8-sig`. Como a coluna de geometria contém textos longos, o DataFrame poderá ocupar mais memória que o tamanho do arquivo no Drive.

In [3]:
df = pd.read_csv(
    ARQUIVO_CSV,
    sep=';',
    encoding='utf-8-sig',
    low_memory=False
)

print('Leitura concluída.')
print(f'Quantidade de registros: {len(df):,}'.replace(',', '.'))
print(f'Quantidade de colunas: {df.shape[1]}')

Leitura concluída.
Quantidade de registros: 480.679
Quantidade de colunas: 6


## 4. Conferir colunas e primeiras linhas

In [4]:
print('Colunas encontradas:')
for numero, coluna in enumerate(df.columns, start=1):
    print(f'{numero}. {coluna}')

display(df.head())

Colunas encontradas:
1. ID_GLEBA
2. #REF_BACEN
3. NU_ORDEM
4. NU_INDICE
5. GT_GEOMETRIA
6. ANO_BASE


,ID_GLEBA,#REF_BACEN,NU_ORDEM,NU_INDICE,GT_GEOMETRIA,ANO_BASE
0,1,511405404,1,0,"POLYGON ((-47.521674 -18.527653 0,-47.521697 -18.528289 0,-47.519128 -18.528105 0,-47.519506 -18.526262 0,-47.519955...",2021
1,2,511405405,1,0,"POLYGON ((-47.579356 -18.906067 0,-47.578942 -18.906589 0,-47.578871 -18.907144 0,-47.578482 -18.907021 0,-47.57761 ...",2021
2,3,511405424,1,0,"POLYGON ((-44.441404 -16.832015 0,-44.441419 -16.831524 0,-44.440824 -16.831596 0,-44.439808999999897 -16.831699 0,-...",2021
3,4,511405446,1,0,"POLYGON ((-44.965694 -20.079657 0,-44.967058 -20.077494 0,-44.968458 -20.078419 0,-44.970597 -20.078106 0,-44.972607...",2021
4,5,511405448,1,0,"POLYGON ((-45.075154 -18.327891 0,-45.077628 -18.329547 0,-45.078869 -18.329854 0,-45.079659 -18.329899 0,-45.08048 ...",2021


## 5. Verificar a coluna ID_GLEBA

Esta célula apenas confere a existência e o preenchimento da coluna. Ela não cria nem modifica identificadores.

In [5]:
if 'ID_GLEBA' in df.columns:
    total_ids = df['ID_GLEBA'].notna().sum()
    ids_distintos = df['ID_GLEBA'].nunique(dropna=True)
    ids_duplicados = df['ID_GLEBA'].duplicated(keep=False).sum()

    print('A coluna ID_GLEBA existe no arquivo.')
    print(f'IDs preenchidos: {total_ids:,}'.replace(',', '.'))
    print(f'IDs distintos: {ids_distintos:,}'.replace(',', '.'))
    print(f'Linhas com ID repetido: {ids_duplicados:,}'.replace(',', '.'))
    print(f'Primeiro ID: {df["ID_GLEBA"].min()}')
    print(f'Último ID: {df["ID_GLEBA"].max()}')
else:
    print('A coluna ID_GLEBA NÃO existe no arquivo.')
    print('Colunas atuais:', list(df.columns))

A coluna ID_GLEBA existe no arquivo.
IDs preenchidos: 480.679
IDs distintos: 480.679
Linhas com ID repetido: 0
Primeiro ID: 1
Último ID: 480679


## 6. Resumo da estrutura e dos valores ausentes

In [6]:
resumo = pd.DataFrame({
    'COLUNA': df.columns,
    'TIPO': df.dtypes.astype(str).values,
    'PREENCHIDOS': df.notna().sum().values,
    'AUSENTES': df.isna().sum().values,
    'VALORES_DISTINTOS': [df[coluna].nunique(dropna=True) for coluna in df.columns]
})

display(resumo)

,COLUNA,TIPO,PREENCHIDOS,AUSENTES,VALORES_DISTINTOS
0,ID_GLEBA,int64,480679,0,480679
1,#REF_BACEN,int64,480679,0,334166
2,NU_ORDEM,int64,480679,0,30
3,NU_INDICE,int64,480679,0,132
4,GT_GEOMETRIA,object,480679,0,395575
5,ANO_BASE,int64,480679,0,5


## 7. Conferir a distribuição por ano

A célula funciona quando a coluna do ano está nomeada como `ANO_BASE`.

In [7]:
if 'ANO_BASE' in df.columns:
    registros_por_ano = (
        df['ANO_BASE']
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis('ANO_BASE')
        .reset_index(name='QUANTIDADE_REGISTROS')
    )
    display(registros_por_ano)
else:
    print('A coluna ANO_BASE não foi encontrada.')

,ANO_BASE,QUANTIDADE_REGISTROS
0,2021,98209
1,2022,84771
2,2023,93911
3,2024,105194
4,2025,98594


## 8. Exemplo de consulta por REF_BACEN

Altere o número abaixo para localizar todos os registros de outro `REF_BACEN`.

In [8]:
REF_BACEN_PESQUISA = 511962772

coluna_ref = '#REF_BACEN' if '#REF_BACEN' in df.columns else 'REF_BACEN'

if coluna_ref in df.columns:
    resultado = df.loc[
        pd.to_numeric(df[coluna_ref], errors='coerce').eq(REF_BACEN_PESQUISA)
    ]
    print(f'Registros encontrados: {len(resultado)}')
    display(resultado)
else:
    print('Não foi localizada uma coluna REF_BACEN no arquivo.')

Registros encontrados: 1


,ID_GLEBA,#REF_BACEN,NU_ORDEM,NU_INDICE,GT_GEOMETRIA,ANO_BASE
22552,22553,511962772,1,0,"POLYGON ((-44.7042 -4.84008 0,-44.7021 -4.84451 0,-44.6976 -4.84037 0,-44.7042 -4.84008 0))",2021
